# Using focal loss due to class imbalance in the MELD dataset


In [1]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import RobertaTokenizer, RobertaModel
from tqdm import tqdm


# 1. IMABlock & Trimodal Architecture
class IMABlock(nn.Module):
    def __init__(self, d_query, d_target, num_heads=8):
        super(IMABlock, self).__init__()
        self.query_proj = nn.Linear(d_query, d_target)
        self.mha = nn.MultiheadAttention(embed_dim=d_target, num_heads=num_heads, batch_first=True)
        self.layer_norm = nn.LayerNorm(d_target)

    def forward(self, query_cls, target_seq):
        q_projected = self.query_proj(query_cls) 
        attn_output, _ = self.mha(query=q_projected, key=target_seq, value=target_seq)
        output = self.layer_norm(q_projected + attn_output)
        return output 

In [2]:
class Trimodal_SSE_FT(nn.Module):
    def __init__(self, num_classes=7, d_speech=768, d_text=1024, d_video=256, fusion_dim=512):
        super(Trimodal_SSE_FT, self).__init__()
        
        # We only need Roberta here, since Audio and Video are pre-extracted!
        self.roberta = RobertaModel.from_pretrained("roberta-large")
        for param in self.roberta.parameters():
            param.requires_grad = False

        self.speech_cls_token = nn.Parameter(torch.randn(1, 1, d_speech))
        self.video_cls_token = nn.Parameter(torch.randn(1, 1, d_video))
        
        s_encoder_layer = nn.TransformerEncoderLayer(d_model=d_speech, nhead=8, batch_first=True)
        self.speech_self_attn = nn.TransformerEncoder(s_encoder_layer, num_layers=1)
        
        v_encoder_layer = nn.TransformerEncoderLayer(d_model=d_video, nhead=8, batch_first=True)
        self.video_self_attn = nn.TransformerEncoder(v_encoder_layer, num_layers=1)

        self.ima_s2t = IMABlock(d_query=d_speech, d_target=d_text)
        self.ima_s2v = IMABlock(d_query=d_speech, d_target=d_video)
        self.ima_t2s = IMABlock(d_query=d_text, d_target=d_speech)
        self.ima_t2v = IMABlock(d_query=d_text, d_target=d_video)
        self.ima_v2s = IMABlock(d_query=d_video, d_target=d_speech)
        self.ima_v2t = IMABlock(d_query=d_video, d_target=d_text)

        self.proj_s2t = nn.Linear(d_text, fusion_dim)
        self.proj_s2v = nn.Linear(d_video, fusion_dim)
        self.proj_t2s = nn.Linear(d_speech, fusion_dim)
        self.proj_t2v = nn.Linear(d_video, fusion_dim)
        self.proj_v2s = nn.Linear(d_speech, fusion_dim)
        self.proj_v2t = nn.Linear(d_text, fusion_dim)

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(fusion_dim * 3, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, pre_extracted_audio, input_text_ids, input_text_mask, pre_extracted_video):
        batch_size = pre_extracted_audio.size(0)

        # 1. Feature Extraction
        speech_out = pre_extracted_audio # Audio is already extracted!
        text_out = self.roberta(input_ids=input_text_ids, attention_mask=input_text_mask).last_hidden_state 
        video_out = pre_extracted_video  # Video is already extracted!

        # 2. Intra-Modal Summarization
        cls_s = self.speech_cls_token.expand(batch_size, -1, -1)
        speech_seq = torch.cat((cls_s, speech_out), dim=1)
        speech_seq = self.speech_self_attn(speech_seq)
        speech_cls = speech_seq[:, 0:1, :] 
        
        cls_v = self.video_cls_token.expand(batch_size, -1, -1)
        video_seq = torch.cat((cls_v, video_out), dim=1)
        video_seq = self.video_self_attn(video_seq)
        video_cls = video_seq[:, 0:1, :] 

        text_cls = text_out[:, 0:1, :] 

        # 3. Inter-Modality Attention (IMA)
        out_s2t = self.ima_s2t(speech_cls, text_out) 
        out_s2v = self.ima_s2v(speech_cls, video_seq) 
        out_t2s = self.ima_t2s(text_cls, speech_seq) 
        out_t2v = self.ima_t2v(text_cls, video_seq) 
        out_v2s = self.ima_v2s(video_cls, speech_seq) 
        out_v2t = self.ima_v2t(video_cls, text_out) 

        # 4. Dimension Projection & Hadamard Products
        speech_final = torch.mul(self.proj_s2t(out_s2t.squeeze(1)), self.proj_s2v(out_s2v.squeeze(1)))
        text_final   = torch.mul(self.proj_t2s(out_t2s.squeeze(1)), self.proj_t2v(out_t2v.squeeze(1)))
        video_final  = torch.mul(self.proj_v2s(out_v2s.squeeze(1)), self.proj_v2t(out_v2t.squeeze(1)))

        # 5. Concatenation & Classification
        combined_features = torch.cat((speech_final, text_final, video_final), dim=1)
        logits = self.classifier(combined_features) 
        return logits

In [3]:
# 2. Dataset & Collate Function
class MeldDataset(Dataset):
    def __init__(self, csv_path, video_dir, audio_dir):
        self.df = pd.read_csv(csv_path)
        self.video_dir = video_dir
        self.audio_dir = audio_dir
        
        # Initialize Text Tokenizer
        self.tokenizer = RobertaTokenizer.from_pretrained('roberta-large')
        
        # MELD Label Mapping
        self.label_map = {'neutral': 0, 'joy': 1, 'sadness': 2, 'anger': 3, 'surprise': 4, 'fear': 5, 'disgust': 6}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        dia_id = row['Dialogue_ID']
        utt_id = row['Utterance_ID']
        filename = f"dia{dia_id}_utt{utt_id}.pt"
        
        video_path = os.path.join(self.video_dir, filename)
        audio_path = os.path.join(self.audio_dir, filename)
        
        # Skip if missing files (due to face extraction failure)
        if not os.path.exists(video_path) or not os.path.exists(audio_path):
            return None

        # Load Pre-Extracted Tensors
        video_tensor = torch.load(video_path, map_location='cpu', weights_only=True) 
        audio_tensor = torch.load(audio_path, map_location='cpu', weights_only=True) 
        
        if audio_tensor.dim() == 3:
            audio_tensor = audio_tensor.squeeze(0) # Remove empty batch dim

        # Process Text on the fly
        text = str(row['Utterance'])
        tokens = self.tokenizer(text, return_tensors='pt', padding='max_length', truncation=True, max_length=128)
        text_ids = tokens['input_ids'].squeeze(0)
        text_mask = tokens['attention_mask'].squeeze(0)
        
        label = self.label_map[row['Emotion']]
        
        return {
            'audio': audio_tensor,
            'text_ids': text_ids,
            'text_mask': text_mask,
            'video': video_tensor,
            'label': torch.tensor(label, dtype=torch.long)
        }

def pad_collate_fn(batch):
    # Filter out missing records
    batch = [b for b in batch if b is not None]
    if len(batch) == 0: return None
    
    audios = [b['audio'] for b in batch]
    text_ids = torch.stack([b['text_ids'] for b in batch])
    text_masks = torch.stack([b['text_mask'] for b in batch])
    videos = [b['video'] for b in batch]
    labels = torch.stack([b['label'] for b in batch])
    
    # Pad variable-length videos/audio with zeros to make them the same length in the batch!
    padded_audios = pad_sequence(audios, batch_first=True)
    padded_videos = pad_sequence(videos, batch_first=True)
    
    return padded_audios, text_ids, text_masks, padded_videos, labels

In [4]:
# 3. Focal Loss 
import numpy as np
import torch.nn.functional as F

# The custom Focal Loss function
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha # The calculated Class weights
        self.gamma = gamma # Focusing parameter (2.0 is standard)

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-F.cross_entropy(inputs, targets, reduction='none'))
        # The focal loss formula!
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


In [ ]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n🚀 Initializing Trimodal Training on {device}...")
    
    model = Trimodal_SSE_FT(num_classes=7).to(device)
    
    print("Loading Training Dataset...")
    train_dataset = MeldDataset(
        csv_path=r"D:\Emotion\MELD.Raw\MELD.Raw\train_sent_emo.csv",
        video_dir=r"./meld_features/train",
        audio_dir=r"./meld_features_audio/train"
    )
    
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=pad_collate_fn)
    
    # NEW: Calculate Class Weights from the Dataset!
    print("Calculating Class Weights for Focal Loss...")
    class_counts = train_dataset.df['Emotion'].value_counts()
    
    # Ensure they match the 0-6 index order exactly!
    ordered_emotions = ['neutral', 'joy', 'sadness', 'anger', 'surprise', 'fear', 'disgust']
    counts_array = [class_counts.get(emo, 1) for emo in ordered_emotions] # get(emo, 1) prevents divide by zero
    
    # Inverse frequency formula: Weight = Total_Samples / Class_Samples
    total_samples = sum(counts_array)
    weights = [total_samples / c for c in counts_array]
    
    # Convert to PyTorch Tensor and normalize
    class_weights = torch.FloatTensor(weights).to(device)
    class_weights = class_weights / class_weights.sum() 
    print(f"Computed Weights: {class_weights.cpu().numpy()}")
    # ---------------------------------------------------------

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    
    # NEW: Swap standard CrossEntropy for Focal Loss!
    criterion = FocalLoss(alpha=class_weights, gamma=2.0)
    
    EPOCHS = 5
    for epoch in range(EPOCHS):
        model.train()
        total_loss, correct, total = 0, 0, 0
        
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        for batch in progress_bar:
            if batch is None: continue
            
            audios, text_ids, text_masks, videos, labels = [b.to(device) for b in batch]
            
            optimizer.zero_grad()
            logits = model(audios, text_ids, text_masks, videos)
            loss = criterion(logits, labels)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            progress_bar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{correct/total:.4f}"})
            
        print(f"Epoch {epoch+1} Complete | Avg Loss: {total_loss/len(train_loader):.4f} | Train Acc: {correct/total:.4f}")

    # Save the trained model!
    torch.save(model.state_dict(), "trimodal_emotion_model_focal.pth")
    print("Model permanently saved as trimodal_emotion_model_focal.pth!")



🚀 Initializing Trimodal Training on cuda...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading Training Dataset...
Calculating Class Weights for Focal Loss...
Computed Weights: [0.01861893 0.05031278 0.12839705 0.07907591 0.07277608 0.32722083
 0.32359847]


Epoch 1/5:   0%|          | 0/1249 [00:00<?, ?it/s]c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\models\roberta\modeling_roberta.py:370: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Epoch 1/5: 100%|██████████| 1249/1249 [12:11<00:00,  1.71it/s, loss=0.0717, acc=0.1887]


Epoch 1 Complete | Avg Loss: 0.0861 | Train Acc: 0.1887


...

Epoch 3 Complete | Avg Loss: 0.0674 | Train Acc: 0.4384


Epoch 4/5: 100%|██████████| 1249/1249 [09:47<00:00,  2.13it/s, loss=0.2210, acc=0.4719]


Epoch 4 Complete | Avg Loss: 0.0634 | Train Acc: 0.4719


Epoch 5/5: 100%|██████████| 1249/1249 [11:32<00:00,  1.80it/s, loss=0.0557, acc=0.4706]


Epoch 5 Complete | Avg Loss: 0.0619 | Train Acc: 0.4706
Model permanently saved as trimodal_emotion_model_focal.pth!


In [ ]:

# RESUME TRAINING: (Load & Train 10 More Epochs)
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Resuming Trimodal Training on {device}...")

# 1. Initialize Blank Model & Load Saved Brain
model = Trimodal_SSE_FT(num_classes=7).to(device)
saved_path = "trimodal_emotion_model_focal.pth"
print(f"Loading weights from {saved_path}...")
model.load_state_dict(torch.load(saved_path, map_location=device))

# 2. Setup Dataset
print("Loading Training Dataset...")
train_dataset = MeldDataset(
    csv_path=r"D:\Emotion\MELD.Raw\MELD.Raw\train_sent_emo.csv",
    video_dir=r"./meld_features/train",
    audio_dir=r"./meld_features_audio/train"
)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=pad_collate_fn)

# 3. Setup Focal Loss & Optimizer
# (Using the class_weights and FocalLoss defined in the cell above!)
criterion = FocalLoss(alpha=class_weights, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5) # Fine-tuning learning rate

# 4. Train for 10 MORE Epochs
ADDITIONAL_EPOCHS = 10
for epoch in range(ADDITIONAL_EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    progress_bar = tqdm(train_loader, desc=f"Resume Epoch {epoch+1}/{ADDITIONAL_EPOCHS}")
    for batch in progress_bar:
        if batch is None: continue
        
        audios, text_ids, text_masks, videos, labels = [b.to(device) for b in batch]
        
        optimizer.zero_grad()
        logits = model(audios, text_ids, text_masks, videos)
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{correct/total:.4f}"})
        
    print(f"Resume Epoch {epoch+1} Complete | Avg Loss: {total_loss/len(train_loader):.4f} | Train Acc: {correct/total:.4f}")

# 5. Overwrite the same save file!
torch.save(model.state_dict(), saved_path)
print(f"\nModel successfully saved back to {saved_path}!")


🚀 Resuming Trimodal Training on cuda...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading weights from trimodal_emotion_model_focal.pth...
Loading Training Dataset...


Resume Epoch 1/10: 100%|██████████| 1249/1249 [10:42<00:00,  1.94it/s, loss=0.0171, acc=0.4875]


Resume Epoch 1 Complete | Avg Loss: 0.0582 | Train Acc: 0.4875


...

Resume Epoch 8 Complete | Avg Loss: 0.0464 | Train Acc: 0.5279


Resume Epoch 9/10: 100%|██████████| 1249/1249 [10:38<00:00,  1.96it/s, loss=0.0168, acc=0.5377]


Resume Epoch 9 Complete | Avg Loss: 0.0446 | Train Acc: 0.5377


Resume Epoch 10/10: 100%|██████████| 1249/1249 [10:39<00:00,  1.95it/s, loss=0.0061, acc=0.5456]


Resume Epoch 10 Complete | Avg Loss: 0.0426 | Train Acc: 0.5456

Model successfully saved back to trimodal_emotion_model_focal.pth!


In [ ]:
# FINAL EVALUATION: VALIDATION & TEST SETS
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load the fully trained model
print("Loading the fully trained model...")
eval_model = Trimodal_SSE_FT(num_classes=7).to(device)

saved_path = "trimodal_emotion_model_focal.pth" 
eval_model.load_state_dict(torch.load(saved_path, map_location=device))

# CRITICAL: Switch to evaluation mode to turn off Dropout and lock weights!
eval_model.eval() 

# 2. Setup Datasets
print("\nLoading Validation (Dev) and Test Datasets...")
dev_dataset = MeldDataset(
    csv_path=r"D:\Emotion\MELD.Raw\MELD.Raw\dev_sent_emo.csv",
    video_dir=r"./meld_features/dev",
    audio_dir=r"./meld_features_audio/dev"
)
# Batch size is 16 here because we aren't using VRAM for training gradients!
dev_loader = DataLoader(dev_dataset, batch_size=16, shuffle=False, collate_fn=pad_collate_fn)

test_dataset = MeldDataset(
    csv_path=r"D:\Emotion\MELD.Raw\MELD.Raw\test_sent_emo.csv",
    video_dir=r"./meld_features/test",
    audio_dir=r"./meld_features_audio/test"
)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=pad_collate_fn)

target_names = ['neutral', 'joy', 'sadness', 'anger', 'surprise', 'fear', 'disgust']

# 3. Reusable Evaluation Function
def evaluate_split(loader, split_name):
    print(f"\n--- Starting Evaluation on {split_name.upper()} Set ---")
    correct, total = 0, 0
    all_preds, all_labels = [], []
    
    with torch.no_grad(): # Tells PyTorch NOT to learn/update weights
        for batch in tqdm(loader, desc=f"Evaluating {split_name}"):
            if batch is None: continue
            
            audios, text_ids, text_masks, videos, labels = [b.to(device) for b in batch]
            
            logits = eval_model(audios, text_ids, text_masks, videos)
            preds = torch.argmax(logits, dim=1)
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print(f"\nFinal {split_name.upper()} Accuracy: {correct/total * 100:.2f}%")
    print(f"\nDetailed {split_name.upper()} Report:")
    print(classification_report(all_labels, all_preds, target_names=target_names, zero_division=0))

# 4. Run the Evaluations!
evaluate_split(dev_loader, "Validation (Dev)")
evaluate_split(test_loader, "Test")


Loading the fully trained model...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Loading Validation (Dev) and Test Datasets...

--- Starting Evaluation on VALIDATION (DEV) Set ---


Evaluating Validation (Dev): 100%|██████████| 70/70 [02:24<00:00,  2.07s/it]



Final VALIDATION (DEV) Accuracy: 38.81%

Detailed VALIDATION (DEV) Report:
              precision    recall  f1-score   support

     neutral       0.69      0.52      0.60       469
         joy       0.36      0.39      0.38       163
     sadness       0.27      0.37      0.31       111
       anger       0.34      0.37      0.35       153
    surprise       0.67      0.12      0.20       150
        fear       0.05      0.03      0.03        40
     disgust       0.02      0.23      0.04        22

    accuracy                           0.39      1108
   macro avg       0.34      0.29      0.27      1108
weighted avg       0.51      0.39      0.42      1108


--- Starting Evaluation on TEST Set ---


Evaluating Test: 100%|██████████| 164/164 [08:15<00:00,  3.02s/it] 


Final TEST Accuracy: 43.87%

Detailed TEST Report:
              precision    recall  f1-score   support

     neutral       0.73      0.56      0.63      1256
         joy       0.41      0.42      0.42       402
     sadness       0.24      0.31      0.27       208
       anger       0.37      0.39      0.38       345
    surprise       0.85      0.14      0.24       281
        fear       0.08      0.08      0.08        50
     disgust       0.06      0.43      0.10        68

    accuracy                           0.44      2610
   macro avg       0.39      0.33      0.30      2610
weighted avg       0.58      0.44      0.47      2610



In [ ]:
# RESUME TRAINING: (Load & Train 15 More Epochs)

import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Resuming Trimodal Training on {device}...")

# 1. Initialize Blank Model & Load Saved Brain
model = Trimodal_SSE_FT(num_classes=7).to(device)
saved_path = "trimodal_emotion_model_focal.pth"
print(f"Loading weights from {saved_path}...")
model.load_state_dict(torch.load(saved_path, map_location=device))

# 2. Setup Dataset
print("Loading Training Dataset...")
train_dataset = MeldDataset(
    csv_path=r"D:\Emotion\MELD.Raw\MELD.Raw\train_sent_emo.csv",
    video_dir=r"./meld_features/train",
    audio_dir=r"./meld_features_audio/train"
)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=pad_collate_fn)

print("Calculating Class Weights for Focal Loss...")
class_counts = train_dataset.df['Emotion'].value_counts()

# Ensure they match the 0-6 index order exactly!
ordered_emotions = ['neutral', 'joy', 'sadness', 'anger', 'surprise', 'fear', 'disgust']
counts_array = [class_counts.get(emo, 1) for emo in ordered_emotions] # get(emo, 1) prevents divide by zero

# Inverse frequency formula: Weight = Total_Samples / Class_Samples
total_samples = sum(counts_array)
weights = [total_samples / c for c in counts_array]

# Convert to PyTorch Tensor and normalize
class_weights = torch.FloatTensor(weights).to(device)
class_weights = class_weights / class_weights.sum() 
print(f"Computed Weights: {class_weights.cpu().numpy()}")

# 3. Setup Focal Loss & Optimizer
# (Using the class_weights and FocalLoss you defined in the cell above!)
criterion = FocalLoss(alpha=class_weights, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5) # Fine-tuning learning rate

# 4. Train for 10 MORE Epochs
ADDITIONAL_EPOCHS = 15
for epoch in range(ADDITIONAL_EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    progress_bar = tqdm(train_loader, desc=f"Resume Epoch {epoch+1}/{ADDITIONAL_EPOCHS}")
    for batch in progress_bar:
        if batch is None: continue
        
        audios, text_ids, text_masks, videos, labels = [b.to(device) for b in batch]
        
        optimizer.zero_grad()
        logits = model(audios, text_ids, text_masks, videos)
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{correct/total:.4f}"})
        
    print(f"Resume Epoch {epoch+1} Complete | Avg Loss: {total_loss/len(train_loader):.4f} | Train Acc: {correct/total:.4f}")

# 5. Overwrite the same save file!
torch.save(model.state_dict(), saved_path)
print(f"\nModel successfully saved back to {saved_path}!")


🚀 Resuming Trimodal Training on cuda...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading weights from trimodal_emotion_model_focal.pth...
Loading Training Dataset...
Calculating Class Weights for Focal Loss...
Computed Weights: [0.01861893 0.05031278 0.12839705 0.07907591 0.07277608 0.32722083
 0.32359847]


Resume Epoch 1/15:   0%|          | 0/1249 [00:00<?, ?it/s]c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\models\roberta\modeling_roberta.py:370: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Resume Epoch 1/15: 100%|██████████| 1249/1249 [13:36<00:00,  1.53it/s, loss=0.0515, acc=0.5574]


Resume Epoch 1 Complete | Avg Loss: 0.0407 | Train Acc: 0.5574


...

Resume Epoch 13 Complete | Avg Loss: 0.0211 | Train Acc: 0.6707


Resume Epoch 14/15: 100%|██████████| 1249/1249 [09:21<00:00,  2.22it/s, loss=0.0177, acc=0.6824]


Resume Epoch 14 Complete | Avg Loss: 0.0196 | Train Acc: 0.6824


Resume Epoch 15/15: 100%|██████████| 1249/1249 [09:24<00:00,  2.21it/s, loss=0.0059, acc=0.6886]


Resume Epoch 15 Complete | Avg Loss: 0.0189 | Train Acc: 0.6886

Model successfully saved back to trimodal_emotion_model_focal.pth!


In [ ]:
# FINAL EVALUATION: VALIDATION & TEST SETS
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load the fully trained model
print("Loading the fully trained model...")
eval_model = Trimodal_SSE_FT(num_classes=7).to(device)

# Make sure you change this to the exact name you used when saving!
saved_path = "trimodal_emotion_model_focal.pth" 
eval_model.load_state_dict(torch.load(saved_path, map_location=device))

# CRITICAL: Switch to evaluation mode to turn off Dropout and lock weights!
eval_model.eval() 

# 2. Setup Datasets
print("\nLoading Validation (Dev) and Test Datasets...")
dev_dataset = MeldDataset(
    csv_path=r"D:\Emotion\MELD.Raw\MELD.Raw\dev_sent_emo.csv",
    video_dir=r"./meld_features/dev",
    audio_dir=r"./meld_features_audio/dev"
)
dev_loader = DataLoader(dev_dataset, batch_size=16, shuffle=False, collate_fn=pad_collate_fn)

test_dataset = MeldDataset(
    csv_path=r"D:\Emotion\MELD.Raw\MELD.Raw\test_sent_emo.csv",
    video_dir=r"./meld_features/test",
    audio_dir=r"./meld_features_audio/test"
)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=pad_collate_fn)

target_names = ['neutral', 'joy', 'sadness', 'anger', 'surprise', 'fear', 'disgust']

# 3. Reusable Evaluation Function
def evaluate_split(loader, split_name):
    print(f"\n--- Starting Evaluation on {split_name.upper()} Set ---")
    correct, total = 0, 0
    all_preds, all_labels = [], []
    
    with torch.no_grad(): # Tells PyTorch NOT to learn/update weights
        for batch in tqdm(loader, desc=f"Evaluating {split_name}"):
            if batch is None: continue
            
            audios, text_ids, text_masks, videos, labels = [b.to(device) for b in batch]
            
            logits = eval_model(audios, text_ids, text_masks, videos)
            preds = torch.argmax(logits, dim=1)
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print(f"\nFinal {split_name.upper()} Accuracy: {correct/total * 100:.2f}%")
    print(f"\nDetailed {split_name.upper()} Report:")
    print(classification_report(all_labels, all_preds, target_names=target_names, zero_division=0))

# 4. Run the Evaluations!
evaluate_split(dev_loader, "Validation (Dev)")
evaluate_split(test_loader, "Test")


Loading the fully trained model...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Loading Validation (Dev) and Test Datasets...

--- Starting Evaluation on VALIDATION (DEV) Set ---


Evaluating Validation (Dev): 100%|██████████| 70/70 [02:50<00:00,  2.43s/it]



Final VALIDATION (DEV) Accuracy: 43.77%

Detailed VALIDATION (DEV) Report:
              precision    recall  f1-score   support

     neutral       0.64      0.58      0.61       469
         joy       0.33      0.48      0.39       163
     sadness       0.28      0.26      0.27       111
       anger       0.30      0.39      0.34       153
    surprise       0.45      0.27      0.34       150
        fear       0.15      0.07      0.10        40
     disgust       0.08      0.09      0.09        22

    accuracy                           0.44      1108
   macro avg       0.32      0.31      0.30      1108
weighted avg       0.45      0.44      0.44      1108


--- Starting Evaluation on TEST Set ---


Evaluating Test: 100%|██████████| 164/164 [10:38<00:00,  3.89s/it] 



Final TEST Accuracy: 47.47%

Detailed TEST Report:
              precision    recall  f1-score   support

     neutral       0.70      0.60      0.65      1256
         joy       0.35      0.51      0.42       402
     sadness       0.28      0.25      0.26       208
       anger       0.31      0.41      0.35       345
    surprise       0.47      0.25      0.32       281
        fear       0.03      0.04      0.03        50
     disgust       0.12      0.12      0.12        68

    accuracy                           0.47      2610
   macro avg       0.32      0.31      0.31      2610
weighted avg       0.51      0.47      0.48      2610



# This is the best general model. 

# After training for 60 whole epochs

In [ ]:
# RESUME TRAINING: (Load & Train 15 More Epochs)

import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Resuming Trimodal Training on {device}...")

# 1. Initialize Blank Model & Load Saved Brain
model = Trimodal_SSE_FT(num_classes=7).to(device)
saved_path = "trimodal_emotion_model_focal.pth"
print(f"Loading weights from {saved_path}...")
model.load_state_dict(torch.load(saved_path, map_location=device))

# 2. Setup Dataset
print("Loading Training Dataset...")
train_dataset = MeldDataset(
    csv_path=r"D:\Emotion\MELD.Raw\MELD.Raw\train_sent_emo.csv",
    video_dir=r"./meld_features/train",
    audio_dir=r"./meld_features_audio/train"
)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=pad_collate_fn)

print("Calculating Class Weights for Focal Loss...")
class_counts = train_dataset.df['Emotion'].value_counts()

# Ensure they match the 0-6 index order exactly!
ordered_emotions = ['neutral', 'joy', 'sadness', 'anger', 'surprise', 'fear', 'disgust']
counts_array = [class_counts.get(emo, 1) for emo in ordered_emotions] # get(emo, 1) prevents divide by zero

# Inverse frequency formula: Weight = Total_Samples / Class_Samples
total_samples = sum(counts_array)
weights = [total_samples / c for c in counts_array]

# Convert to PyTorch Tensor and normalize
class_weights = torch.FloatTensor(weights).to(device)
class_weights = class_weights / class_weights.sum() 
print(f"Computed Weights: {class_weights.cpu().numpy()}")

# 3. Setup Focal Loss & Optimizer
criterion = FocalLoss(alpha=class_weights, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5) # Fine-tuning learning rate

# 4. Train for 10 MORE Epochs
ADDITIONAL_EPOCHS = 30
for epoch in range(ADDITIONAL_EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    progress_bar = tqdm(train_loader, desc=f"Resume Epoch {epoch+1}/{ADDITIONAL_EPOCHS}")
    for batch in progress_bar:
        if batch is None: continue
        
        audios, text_ids, text_masks, videos, labels = [b.to(device) for b in batch]
        
        optimizer.zero_grad()
        logits = model(audios, text_ids, text_masks, videos)
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{correct/total:.4f}"})
        
    print(f"Resume Epoch {epoch+1} Complete | Avg Loss: {total_loss/len(train_loader):.4f} | Train Acc: {correct/total:.4f}")


# 5. Overwrite the same save file!
torch.save(model.state_dict(), saved_path)
print(f"\nModel successfully saved back to {saved_path}!")



🚀 Resuming Trimodal Training on cuda...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading weights from trimodal_emotion_model_focal.pth...
Loading Training Dataset...
Calculating Class Weights for Focal Loss...
Computed Weights: [0.01861893 0.05031278 0.12839705 0.07907591 0.07277608 0.32722083
 0.32359847]


Resume Epoch 1/30:   0%|          | 0/1249 [00:00<?, ?it/s]c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\models\roberta\modeling_roberta.py:370: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Resume Epoch 1/30: 100%|██████████| 1249/1249 [15:19<00:00,  1.36it/s, loss=0.0185, acc=0.7007]


Resume Epoch 1 Complete | Avg Loss: 0.0178 | Train Acc: 0.7007


...

Resume Epoch 28 Complete | Avg Loss: 0.0048 | Train Acc: 0.8777


Resume Epoch 29/30: 100%|██████████| 1249/1249 [08:55<00:00,  2.33it/s, loss=0.0028, acc=0.8844]


Resume Epoch 29 Complete | Avg Loss: 0.0044 | Train Acc: 0.8844


Resume Epoch 30/30: 100%|██████████| 1249/1249 [08:49<00:00,  2.36it/s, loss=0.0008, acc=0.8766]


Resume Epoch 30 Complete | Avg Loss: 0.0053 | Train Acc: 0.8766

Model successfully saved back to trimodal_emotion_model_focal.pth!


In [5]:
# FINAL EVALUATION: VALIDATION & TEST SETS
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load the fully trained model
print("Loading the fully trained model...")
eval_model = Trimodal_SSE_FT(num_classes=7).to(device)

# Make sure you change this to the exact name you used when saving!
saved_path = "trimodal_emotion_model_focal.pth" 
eval_model.load_state_dict(torch.load(saved_path, map_location=device))

# CRITICAL: Switch to evaluation mode to turn off Dropout and lock weights!
eval_model.eval() 

# 2. Setup Datasets
print("\nLoading Validation (Dev) and Test Datasets...")
dev_dataset = MeldDataset(
    csv_path=r"D:\Emotion\MELD.Raw\MELD.Raw\dev_sent_emo.csv",
    video_dir=r"./meld_features/dev",
    audio_dir=r"./meld_features_audio/dev"
)
dev_loader = DataLoader(dev_dataset, batch_size=16, shuffle=False, collate_fn=pad_collate_fn)

test_dataset = MeldDataset(
    csv_path=r"D:\Emotion\MELD.Raw\MELD.Raw\test_sent_emo.csv",
    video_dir=r"./meld_features/test",
    audio_dir=r"./meld_features_audio/test"
)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=pad_collate_fn)

target_names = ['neutral', 'joy', 'sadness', 'anger', 'surprise', 'fear', 'disgust']

# 3. Reusable Evaluation Function
def evaluate_split(loader, split_name):
    print(f"\n--- Starting Evaluation on {split_name.upper()} Set ---")
    correct, total = 0, 0
    all_preds, all_labels = [], []
    
    with torch.no_grad(): # Tells PyTorch NOT to learn/update weights
        for batch in tqdm(loader, desc=f"Evaluating {split_name}"):
            if batch is None: continue
            
            audios, text_ids, text_masks, videos, labels = [b.to(device) for b in batch]
            
            logits = eval_model(audios, text_ids, text_masks, videos)
            preds = torch.argmax(logits, dim=1)
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print(f"\nFinal {split_name.upper()} Accuracy: {correct/total * 100:.2f}%")
    print(f"\nDetailed {split_name.upper()} Report:")
    print(classification_report(all_labels, all_preds, target_names=target_names, zero_division=0))

# 4. Run the Evaluations!
evaluate_split(dev_loader, "Validation (Dev)")
evaluate_split(test_loader, "Test")


Loading the fully trained model...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Loading Validation (Dev) and Test Datasets...

--- Starting Evaluation on VALIDATION (DEV) Set ---


Evaluating Validation (Dev):   0%|          | 0/70 [00:00<?, ?it/s]c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\models\roberta\modeling_roberta.py:370: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Evaluating Validation (Dev): 100%|██████████| 70/70 [00:45<00:00,  1.54it/s]



Final VALIDATION (DEV) Accuracy: 38.54%

Detailed VALIDATION (DEV) Report:
              precision    recall  f1-score   support

     neutral       0.63      0.51      0.56       469
         joy       0.25      0.45      0.32       163
     sadness       0.33      0.23      0.27       111
       anger       0.35      0.22      0.27       153
    surprise       0.30      0.34      0.32       150
        fear       0.14      0.03      0.04        40
     disgust       0.03      0.14      0.06        22

    accuracy                           0.39      1108
   macro avg       0.29      0.27      0.26      1108
weighted avg       0.43      0.39      0.40      1108


--- Starting Evaluation on TEST Set ---


Evaluating Test: 100%|██████████| 164/164 [05:15<00:00,  1.92s/it] 



Final TEST Accuracy: 40.15%

Detailed TEST Report:
              precision    recall  f1-score   support

     neutral       0.66      0.49      0.56      1256
         joy       0.27      0.52      0.35       402
     sadness       0.20      0.16      0.18       208
       anger       0.35      0.22      0.27       345
    surprise       0.32      0.35      0.34       281
        fear       0.00      0.00      0.00        50
     disgust       0.07      0.22      0.11        68

    accuracy                           0.40      2610
   macro avg       0.27      0.28      0.26      2610
weighted avg       0.46      0.40      0.42      2610



# Therefore the best model which generalized was created after training around 30 epochs and, thereafter the model began overfitting. 